# Week-08 · Chapter 07 — Deep Learning & Generative AI

이번 Notebook은 **독립적으로 실행할 수 있는 Colab 실습 자료**입니다. 수업 본문은 `README.md`를 참고하세요.

- 7.1: NumPy 기반 순전파 계산
- 7.2: PyTorch 기반 Digits(8×8) 숫자 분류 학습
- 7.3: MobileNetV3 사전 학습 이미지 분류
- 7.4: SmolLM2 소형 언어 모델 텍스트 생성

**7.3·7.4는 최초 실행 시 인터넷으로 모델 가중치를 다운로드합니다.** GPU가 없어도 7.1·7.2는 CPU로 실습할 수 있습니다.

## 실행 환경 준비

기본 라이브러리를 설치합니다. Colab에 이미 설치되어 있다면 재설치할 필요는 없습니다. GPU 런타임을 선택해도 CUDA 지원 빌드가 없으면 CPU로 동작할 수 있습니다.

In [ ]:
%pip install -q numpy matplotlib scikit-learn pillow transformers

In [ ]:
import torch
import sklearn
import transformers
try:
    import torchvision
    tv = torchvision.__version__
except ImportError:
    tv = "설치 필요 (7.3에서 사용)"

print("PyTorch:", torch.__version__)
print("TorchVision:", tv)
print("scikit-learn:", sklearn.__version__)
print("Transformers:", transformers.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

> `torch` 또는 `torchvision`이 없다면 별도 셀에서 `%pip install torch torchvision`을 실행하세요. PyTorch와 TorchVision은 호환되는 버전으로 설치해야 합니다. 패키지 설치 중 런타임 재시작 안내가 나오면 재시작 후 처음부터 실행하세요.

## 7.1. Introduction to Deep Learning

입력 두 개, 은닉 노드 세 개, 출력 한 개로 이루어진 간단한 신경망을 계산합니다. **고정 가중치를 사용하므로 아직 학습하는 실습은 아닙니다.**

In [ ]:
"""7.1 — NumPy로 인공신경망 순전파 확인 (고정 가중치, 학습 없음)."""
import numpy as np

# 입력 특징 2개 (학습용 가상 입력)
x = np.array([0.6, 0.8])

# 입력층 2 → 은닉층 3 → 출력층 1
W1 = np.array([[0.4, -0.3, 0.7], [0.2, 0.6, -0.1]])
b1 = np.array([0.1, 0.0, 0.2])
W2 = np.array([[0.5], [-0.4], [0.8]])
b2 = np.array([0.1])

z = x @ W1 + b1
hidden = np.maximum(z, 0)  # ReLU: 음수는 0으로, 양수는 그대로
output = hidden @ W2 + b2

print("입력:", x)
print("은닉층 가중합:", np.round(z, 3))
print("은닉층 ReLU 출력:", np.round(hidden, 3))
print("최종 출력:", np.round(output, 3))
print("참고: 이 예제는 고정 가중치로 계산만 수행하며, 예측 모델을 학습하지 않습니다.")


## 7.2. Neural Networks & Model Training

`scikit-learn`의 **Digits 데이터셋(8×8, 1,797개)**을 이용하여 숫자 0~9를 구분하는 신경망을 학습합니다.

- `EPOCHS`를 변경하며 Train Loss와 Test Accuracy 비교
- `HIDDEN_UNITS`를 변경하며 모델 구조 비교
- 학습 곡선과 실제 테스트 숫자 이미지 출력

In [ ]:
"""7.2 — PyTorch로 scikit-learn Digits(8×8) 분류 학습.

별도 데이터 파일 불필요. CPU 실행 가능. 실행 후 이미지 두 개를 assets/에 저장합니다.
"""
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
EPOCHS = 15            # 학습 반복 횟수: 5, 15, 30으로 바꾸어 보세요.
HIDDEN_UNITS = 64       # 은닉층 노드: 16, 64, 128을 비교해 보세요.
BATCH_SIZE = 64
LEARNING_RATE = 0.01

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(2, torch.get_num_threads()))

# Notebook에서는 assets/, Python 파일 실행 시에는 Week-08/assets/ 사용
out_dir = (Path(__file__).resolve().parents[1] / "assets") if "__file__" in globals() else Path("assets")
out_dir.mkdir(parents=True, exist_ok=True)

digits = load_digits()
X = (digits.data / 16.0).astype(np.float32)  # 8x8 → 특징 64개, 픽셀값 0~1
y = digits.target.astype(np.int64)          # 정답: 0~9
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE, shuffle=True
)

model = nn.Sequential(
    nn.Linear(64, HIDDEN_UNITS),
    nn.ReLU(),
    nn.Linear(HIDDEN_UNITS, 10),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

loss_history = []
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for features, labels in train_loader:
        optimizer.zero_grad()
        logits = model(features)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    train_loss = total_loss / len(X_train)
    loss_history.append(train_loss)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f}")

model.eval()
with torch.inference_mode():
    predictions = model(torch.from_numpy(X_test)).argmax(dim=1).numpy()
accuracy = float(np.mean(predictions == y_test))
print(f"Test Accuracy: {accuracy:.1%} ({np.sum(predictions == y_test)}/{len(y_test)})")

# Figure 1 — 학습 손실 곡선
fig, ax = plt.subplots(figsize=(9.6, 5.4), dpi=170)
ax.plot(range(1, EPOCHS+1), loss_history, marker="o", linewidth=2.4, markersize=5)
ax.set_title("Training Loss by Epoch", fontsize=16, pad=16)
ax.set_xlabel("Epoch")
ax.set_ylabel("Train Loss")
ax.set_xticks(range(1, EPOCHS+1))
ax.grid(alpha=0.22)
fig.tight_layout()
fig.savefig(out_dir / "7_2_training_curve.png", facecolor="white", dpi=170)
plt.show()

# Figure 2 — 테스트 이미지와 예측 숫자 비교
fig, axes = plt.subplots(3, 4, figsize=(10, 8.1), dpi=170)
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r", interpolation="nearest")
    correct = predictions[i] == y_test[i]
    ax.set_title(f"True {y_test[i]}  /  Pred {predictions[i]}" + ("" if correct else "  *"), fontsize=10)
    ax.axis("off")
fig.suptitle("Handwritten Digit Classification — Test Samples", fontsize=15)
fig.tight_layout(pad=1.9, h_pad=2.2)
fig.savefig(out_dir / "7_2_digits_predictions.png", facecolor="white", dpi=170)
plt.show()


## 7.3. Computer Vision & Image Recognition

**사전 학습된 MobileNetV3-Small**로 사진을 분류합니다. 공개 강아지 샘플 이미지를 자동 다운로드합니다.

- 다른 사진을 넣으려면 `UPLOAD_OWN_IMAGE = True`로 변경하세요.
- 예측한 상위 5개 클래스와 점수를 확인하세요.
- 실행 시 사진과 점수의 그래프가 `assets/7_3_image_prediction.png`에 저장됩니다.

**참고:** 최초 실행 시 모델 가중치를 다운로드해야 하며 네트워크 오류가 발생할 수 있습니다.

In [ ]:
"""7.3 — MobileNetV3-Small 사전 학습 모델을 이용한 이미지 분류.

첫 실행 시 공개 샘플 사진과 pretrained 가중치를 내려받으므로 인터넷 연결이 필요합니다.
UPLOAD_OWN_IMAGE=True로 바꾸면 Colab에서 직접 이미지 파일을 선택할 수 있습니다.
"""
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

UPLOAD_OWN_IMAGE = False
SAMPLE_IMAGE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"

out_dir = (Path(__file__).resolve().parents[1] / "assets") if "__file__" in globals() else Path("assets")
out_dir.mkdir(parents=True, exist_ok=True)

if UPLOAD_OWN_IMAGE:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("사진 업로드는 Google Colab에서 실행하거나 코드를 로컬 경로로 수정하세요.") from exc
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("분류할 이미지를 업로드하세요.")
    img_path = Path(next(iter(uploaded)))
else:
    img_path = out_dir / "sample_dog.jpg"
    if not img_path.is_file():
        print("공개 예제 사진 다운로드 중...")
        urlretrieve(SAMPLE_IMAGE_URL, img_path)

image = Image.open(img_path).convert("RGB")
weights = MobileNet_V3_Small_Weights.DEFAULT
model = mobilenet_v3_small(weights=weights)
model.eval()

input_tensor = weights.transforms()(image).unsqueeze(0)
with torch.inference_mode():
    logits = model(input_tensor)
    scores = logits.softmax(dim=1)[0]
    top_scores, top_ids = torch.topk(scores, k=5)

classes = weights.meta["categories"]
names = [classes[i] for i in top_ids.tolist()]
values = top_scores.tolist()

print("ImageNet Top-5 Prediction")
for rank, (name, score) in enumerate(zip(names, values), 1):
    print(f"{rank}. {name}: {score:.1%}")

fig, (ax_image, ax_score) = plt.subplots(1, 2, figsize=(12.2, 5), dpi=170,
                                        gridspec_kw={"width_ratios": [1, 1.3]})
ax_image.imshow(image)
ax_image.set_title("Input Image")
ax_image.axis("off")
ax_score.barh(names[::-1], [v * 100 for v in values[::-1]])
ax_score.set_title("Top 5 Predictions")
ax_score.set_xlabel("Softmax Score (%)")
ax_score.set_xlim(0, 100)
ax_score.grid(axis="x", alpha=0.18)
ax_score.set_axisbelow(True)
fig.tight_layout()
result = out_dir / "7_3_image_prediction.png"
fig.savefig(result, facecolor="white", dpi=170, bbox_inches="tight")
plt.show()
print("그래프 저장:", result)
print("참고: Softmax 출력은 실제 정답일 확률을 보장하지 않습니다.")


## 7.4. Generative AI & Large Language Models

영어 중심 소형 언어 모델 `HuggingFaceTB/SmolLM2-135M-Instruct`로 텍스트를 생성합니다.

- `PROMPT`를 다른 질문으로 변경하세요.
- `MAX_NEW_TOKENS`를 바꾸어 출력 길이를 비교해 보세요.
- 생성 결과가 모든 조건을 충족하거나 항상 사실이라고 가정하지 마세요.

**참고:** 최초 실행 시 언어 모델을 다운로드하며, CPU에서는 텍스트 생성에 시간이 걸릴 수 있습니다.

In [ ]:
"""7.4 — 공개 소형 언어 모델(SmolLM2-135M-Instruct)을 이용한 텍스트 생성.

처음 실행할 때 Hugging Face에서 가중치를 다운로드하므로 인터넷이 필요합니다.
주로 영어로 학습된 소형 모델이므로 영어 프롬프트를 권장합니다.
"""
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
PROMPT = "List three simple ways to save electricity at home. Keep each tip short."
MAX_NEW_TOKENS = 70

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("첫 실행 시 공개 모델을 다운로드합니다.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
model.eval()

messages = [{"role": "user", "content": PROMPT}]
input_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_tensors="pt"
).to(device)

with torch.inference_mode():
    generated_ids = model.generate(
        input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

new_ids = generated_ids[0][input_ids.shape[-1]:]
answer = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
print("Prompt:", PROMPT)
print("Generated text:\n", answer)
print("주의: 생성 결과는 사실 확인이 필요할 수 있습니다.")


## Exercise

**Exercise 01 — 숫자 모델 비교**  
`practice/exercise_01_digits.py`에서 준비한 Epoch·Hidden Units 다섯 가지 조건을 실행하고 결과를 기록하세요.

**Exercise 02 — 이미지 비교**  
7.3 셀의 `UPLOAD_OWN_IMAGE`를 `True`로 바꿔 공개 사물 사진을 업로드한 후 Top-5 결과를 비교하세요.

**Exercise 03 — Prompt 비교**  
`practice/exercise_02_prompt.py`는 세 가지 Prompt를 `30` 및 `70` 토큰 제한에서 비교하는 독립 실행 코드입니다. Colab에서 ZIP을 업로드했다면 `%run practice/exercise_02_prompt.py`로 실행할 수 있습니다.

자세한 문항은 `README.md`의 Summary → Quiz → Exercise를 참고하세요.